# Scikit-Learn Pipelines

Pipelines chain together multiple steps (like preprocessing and modeling) into a single object. They are an absolute game-changer for writing clean, professional, and robust machine learning code.

## The Problem with Manual Preprocessing (Data Leakage)

When beginners write ML code, they often preprocess their entire dataset *before* splitting it into train and test sets. This causes **Data Leakage**.

If you scale the entire dataset or impute missing values using the mean of the *entire* dataset, information from the test set 'leaks' into the training process. Your model is essentially cheating because it has seen statistics from the data it's supposed to be tested on.

**The Solution**: Pipelines ensure that transformations are only *fit* on the training data, and then *applied* (transformed) to the test data. They completely eliminate this class of data leakage.

## 1. Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 2. Loading the Dataset

In [2]:
# We will use the Titanic dataset
df = pd.read_csv('../titanic.csv')

# Select a subset of features for simplicity
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked']
target = 'Survived'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Defining the Processing Steps (`ColumnTransformer`)

`ColumnTransformer` is a fantastic tool that allows you to apply different preprocessing pipelines to different columns (e.g., scaling numerical features while one-hot encoding categorical features).

In [3]:
# Separate numerical and categorical columns
numeric_features = ['Age', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Numerical Pipeline: Impute missing values (median) -> Scale (StandardScaler)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute missing values (mode) -> One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 4. Creating the Full Pipeline

Now we combine our preprocessor with a machine learning model.

In [4]:
# Combine preprocessing with an estimator
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

## 5. Training and Evaluation

In [5]:
# Fit the pipeline on the training data
# Under the hood, this calls fit_transform() on the preprocessor, then fit() on the classifier
clf.fit(X_train, y_train)

# Predict and evaluate
# Under the hood, this calls transform() on the preprocessor, then predict() on the classifier
y_pred = clf.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')

Accuracy: 0.7821


## 6. Grid Search with Pipelines

Pipelines make it incredibly easy to tune hyperparameters for *both* the preprocessing steps and the model simultaneously, without any data leakage during cross-validation.

In [6]:
# Define the parameter grid
# Note the syntax: stepname__parametername
# We can even test different imputation strategies in our grid search!
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 5]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(clf, param_grid, cv=3, scoring='accuracy')

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
print(f'Best Cross-validation Accuracy: {grid_search.best_score_:.4f}')

# Evaluate on test set
best_model = grid_search.best_estimator_
test_accuracy = accuracy_score(y_test, best_model.predict(X_test))
print(f'Test Accuracy with Best Model: {test_accuracy:.4f}')

Best Parameters: {'classifier__max_depth': 5, 'classifier__n_estimators': 100, 'preprocessor__num__imputer__strategy': 'mean'}
Best Cross-validation Accuracy: 0.8259
Test Accuracy with Best Model: 0.7989


## 7. Adding Custom Transformers to Pipelines

Sometimes you need to apply a custom mathematical function (like log-transforming a skewed feature). You can use `FunctionTransformer` to easily integrate custom logic into your pipeline.

In [7]:
# Custom transformer to apply log1p (log(1+x))
log_transformer = FunctionTransformer(np.log1p, validate=False)

# Let's create a special pipeline just for the heavily skewed 'Fare' column
fare_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log_transform', log_transformer),
    ('scaler', StandardScaler())
])

print('Custom log transformer created successfully. It can now be slotted into a ColumnTransformer!')

Custom log transformer created successfully. It can now be slotted into a ColumnTransformer!


## Summary

- **Pipelines** ensure all transformations are applied consistently to training, validation, and testing data, completely preventing data leakage.
- **ColumnTransformer** is essential for applying different preprocessing steps to different subsets of features.
- **GridSearchCV + Pipelines** allows for comprehensive and safe hyperparameter tuning across the entire ML workflow.
- **FunctionTransformer** allows you to seamlessly integrate your own custom preprocessing functions.